In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Pastikan NLTK data sudah diunduh
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True) # Tambahkan ini untuk mengatasi LookupError

# Dapatkan daftar stop words bahasa Indonesia
stop_words_id = set(stopwords.words('indonesian'))

def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Hapus tanda baca dan karakter non-alfanumerik
    text = re.sub(r'[^a-z\s]', '', text) # Hanya menyisakan huruf kecil dan spasi
    # 3. Tokenisasi
    tokens = word_tokenize(text)
    # 4. Hapus stop words
    filtered_tokens = [word for word in tokens if word not in stop_words_id and len(word) > 1]
    # Gabungkan kembali menjadi string
    return ' '.join(filtered_tokens)

# --- 1. Persiapan Data Simulasi Diagnosa ---
data_diagnosa = {
    'symptom_description': [
        "Pasien mengeluh demam tinggi, batuk kering, dan nyeri tenggorokan selama tiga hari.",
        "Mengalami hidung tersumbat, bersin-bersin, mata gatal setelah terpapar debu.",
        "Merasa lelah terus-menerus, kehilangan minat pada hobi, sulit tidur selama berminggu-minggu.",
        "Sakit kepala hebat, mual, muntah, dan sensitif terhadap cahaya.",
        "Kulit gatal kemerahan, bengkak di beberapa area setelah makan seafood.",
        "Perasaan sedih yang mendalam, putus asa, dan perubahan nafsu makan signifikan.",
        "Gejala mirip flu tapi lebih ringan, dengan pilek dan sedikit demam.",
        "Sering cemas tanpa alasan jelas, detak jantung cepat, tangan berkeringat.",
        "Sakit perut hebat, diare, dan muntah berkali-kali.", # Diare
        "Batuk berdahak, sesak napas, dan demam rendah.", # Bronkitis
        "Gatal-gatal di seluruh tubuh, ruam merah, dan bengkak bibir setelah makan udang.", # Alergi
        "Perasaan panik mendadak, jantung berdebar, dan kesulitan bernapas tanpa sebab jelas.", # Gangguan Kecemasan
        "Demam dan nyeri otot serta sendi, disertai ruam merah.", # Demam Berdarah
        "Sering buang air kecil, mudah haus, dan cepat lapar, berat badan turun tanpa sebab.", # Diabetes
        "Mata kabur, sakit kepala, leher kaku, dan sensitif terhadap cahaya.", # Migrain
        "Tidak bisa tidur, nafsu makan hilang, merasa tidak berharga.", # Depresi
        "Sakit tenggorokan, batuk kering, hidung tersumbat, dan nyeri kepala ringan.", # Flu Biasa
        "Demam tinggi, menggigil, nyeri badan, dan batuk yang parah.", # Influenza
        "Kulit kering bersisik, gatal parah, terutama di malam hari.", # Eksim
        "Nyeri ulu hati, mual, perut kembung, dan sering bersendawa.", # Maag
        "Kesulitan berkonsentrasi, sering lupa, mudah tersinggung.", # Stres
        "Lemas, pucat, pusing, dan detak jantung tidak teratur.", # Anemia
        "Nyeri dada, sesak napas, berkeringat dingin, menjalar ke lengan kiri." # Serangan Jantung (contoh ekstrem)
    ],
    'diagnosis': [
        'influenza',
        'alergi',
        'depresi',
        'migrain',
        'alergi',
        'depresi',
        'flu_biasa',
        'gangguan_kecemasan',
        'diare',
        'bronkitis',
        'alergi',
        'gangguan_kecemasan',
        'demam_berdarah',
        'diabetes',
        'migrain',
        'depresi',
        'flu_biasa',
        'influenza',
        'eksim',
        'maag',
        'stres',
        'anemia',
        'serangan_jantung'
    ]
}

df_diagnosa = pd.DataFrame(data_diagnosa)
print("Dataset untuk Terapi Diagnosa:")
display(df_diagnosa)

# --- 2. Pra-pemrosesan Teks ---
df_diagnosa['cleaned_symptom'] = df_diagnosa['symptom_description'].apply(preprocess_text)
print("\nDataset setelah pra-pemrosesan:")
display(df_diagnosa)

# --- 3. Ekstraksi Fitur (TF-IDF) ---
X_diag = df_diagnosa['cleaned_symptom']
y_diag = df_diagnosa['diagnosis']

# Menghapus 'stratify=y_diag' karena beberapa kelas hanya memiliki satu anggota
X_train_diag, X_test_diag, y_train_diag, y_test_diag = train_test_split(X_diag, y_diag, test_size=0.3, random_state=42)

print(f"\nJumlah data training diagnosa: {len(X_train_diag)}")
print(f"Jumlah data testing diagnosa: {len(X_test_diag)}")

tfidf_vectorizer_diag = TfidfVectorizer()
X_train_tfidf_diag = tfidf_vectorizer_diag.fit_transform(X_train_diag)
X_test_tfidf_diag = tfidf_vectorizer_diag.transform(X_test_diag)

print("Bentuk X_train_tfidf_diag:", X_train_tfidf_diag.shape)
print("Bentuk X_test_tfidf_diag:", X_test_tfidf_diag.shape)

# --- 4. Pelatihan Model (SVM) ---
svm_model_diag = SVC(kernel='linear', random_state=42)
svm_model_diag.fit(X_train_tfidf_diag, y_train_diag)
print("\nModel SVM untuk diagnosa telah berhasil dilatih.")

# --- 5. Evaluasi Model ---
y_pred_diag = svm_model_diag.predict(X_test_tfidf_diag)
accuracy_diag = accuracy_score(y_test_diag, y_pred_diag)
print(f"\nAkurasi Model Diagnosa: {accuracy_diag:.2f}")
report_diag = classification_report(y_test_diag, y_pred_diag, zero_division=0)
print("\nClassification Report Diagnosa:\n", report_diag)

# --- 6. Uji Model dengan Kalimat Input Baru ---
new_symptoms = [
    "Pasien merasa sakit tenggorokan, hidung meler, dan batuk ringan.", # flu_biasa
    "Mual dan pusing sangat parah disertai nyeri kepala di satu sisi.", # migrain
    "Badan pegal-pegal dan demam tinggi.", # influenza
    "Saya merasa sangat sedih dan tidak ingin melakukan apa-apa." # depresi
]

df_new_symptoms = pd.DataFrame({'symptom_description': new_symptoms})
df_new_symptoms['cleaned_symptom'] = df_new_symptoms['symptom_description'].apply(preprocess_text)

X_new_tfidf_diag = tfidf_vectorizer_diag.transform(df_new_symptoms['cleaned_symptom'])
new_predictions_diag = svm_model_diag.predict(X_new_tfidf_diag)

df_new_symptoms['predicted_diagnosis'] = new_predictions_diag

print("\nPrediksi Diagnosa untuk Gejala Baru:")
display(df_new_symptoms)

Dataset untuk Terapi Diagnosa:


,symptom_description,diagnosis
0,"Pasien mengeluh demam tinggi, batuk kering, da...",influenza
1,"Mengalami hidung tersumbat, bersin-bersin, mat...",alergi
2,"Merasa lelah terus-menerus, kehilangan minat p...",depresi
3,"Sakit kepala hebat, mual, muntah, dan sensitif...",migrain
4,"Kulit gatal kemerahan, bengkak di beberapa are...",alergi
5,"Perasaan sedih yang mendalam, putus asa, dan p...",depresi
6,"Gejala mirip flu tapi lebih ringan, dengan pil...",flu_biasa
7,"Sering cemas tanpa alasan jelas, detak jantung...",gangguan_kecemasan
8,"Sakit perut hebat, diare, dan muntah berkali-k...",diare
9,"Batuk berdahak, sesak napas, dan demam rendah.",bronkitis



Dataset setelah pra-pemrosesan:


,symptom_description,diagnosis,cleaned_symptom
0,"Pasien mengeluh demam tinggi, batuk kering, da...",influenza,pasien mengeluh demam batuk kering nyeri tengg...
1,"Mengalami hidung tersumbat, bersin-bersin, mat...",alergi,mengalami hidung tersumbat bersinbersin mata g...
2,"Merasa lelah terus-menerus, kehilangan minat p...",depresi,lelah terusmenerus kehilangan minat hobi sulit...
3,"Sakit kepala hebat, mual, muntah, dan sensitif...",migrain,sakit kepala hebat mual muntah sensitif cahaya
4,"Kulit gatal kemerahan, bengkak di beberapa are...",alergi,kulit gatal kemerahan bengkak area makan seafood
5,"Perasaan sedih yang mendalam, putus asa, dan p...",depresi,perasaan sedih mendalam putus asa perubahan na...
6,"Gejala mirip flu tapi lebih ringan, dengan pil...",flu_biasa,gejala flu ringan pilek demam
7,"Sering cemas tanpa alasan jelas, detak jantung...",gangguan_kecemasan,cemas alasan detak jantung cepat tangan berker...
8,"Sakit perut hebat, diare, dan muntah berkali-k...",diare,sakit perut hebat diare muntah berkalikali
9,"Batuk berdahak, sesak napas, dan demam rendah.",bronkitis,batuk berdahak sesak napas demam rendah



Jumlah data training diagnosa: 16
Jumlah data testing diagnosa: 7
Bentuk X_train_tfidf_diag: (16, 93)
Bentuk X_test_tfidf_diag: (7, 93)

Model SVM untuk diagnosa telah berhasil dilatih.

Akurasi Model Diagnosa: 0.14

Classification Report Diagnosa:
                 precision    recall  f1-score   support

        alergi       0.00      0.00      0.00         1
     bronkitis       0.00      0.00      0.00         1
demam_berdarah       0.00      0.00      0.00         1
       depresi       1.00      1.00      1.00         1
         diare       0.00      0.00      0.00         1
     flu_biasa       0.00      0.00      0.00         0
     influenza       0.00      0.00      0.00         2
       migrain       0.00      0.00      0.00         0

      accuracy                           0.14         7
     macro avg       0.12      0.12      0.12         7
  weighted avg       0.14      0.14      0.14         7


Prediksi Diagnosa untuk Gejala Baru:


,symptom_description,cleaned_symptom,predicted_diagnosis
0,"Pasien merasa sakit tenggorokan, hidung meler,...",pasien sakit tenggorokan hidung meler batuk ri...,flu_biasa
1,Mual dan pusing sangat parah disertai nyeri ke...,mual pusing parah disertai nyeri kepala sisi,migrain
2,Badan pegal-pegal dan demam tinggi.,badan pegalpegal demam,flu_biasa
3,Saya merasa sangat sedih dan tidak ingin melak...,sedih apaapa,depresi


In [2]:
import joblib

# Tentukan nama file untuk menyimpan model dan vectorizer diagnostik
model_diag_filename = 'svm_diagnosa_model.joblib'
vectorizer_diag_filename = 'tfidf_diagnosa_vectorizer.joblib'

# Simpan model SVM untuk diagnosa
joblib.dump(svm_model_diag, model_diag_filename)
print(f"Model SVM Diagnosa telah disimpan ke '{model_diag_filename}'")

# Simpan TF-IDF Vectorizer untuk diagnosa
joblib.dump(tfidf_vectorizer_diag, vectorizer_diag_filename)
print(f"TF-IDF Vectorizer Diagnosa telah disimpan ke '{vectorizer_diag_filename}'")

Model SVM Diagnosa telah disimpan ke 'svm_diagnosa_model.joblib'
TF-IDF Vectorizer Diagnosa telah disimpan ke 'tfidf_diagnosa_vectorizer.joblib'


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output

# Pastikan NLTK data sudah diunduh
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# --- Definisi fungsi pra-pemrosesan teks (re-defining to ensure scope) ---
stop_words_id = set(stopwords.words('indonesian'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text) # Hanya menyisakan huruf kecil dan spasi
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words_id and len(word) > 1]
    return ' '.join(filtered_tokens)

# --- Fungsi untuk menghasilkan Terapi berdasarkan Diagnosis ---
def generate_therapy(predicted_diagnosis, original_symptom_description):
    therapy_suggestions = {
        'influenza': "Istirahat cukup, minum banyak cairan, dan konsumsi obat pereda demam/nyeri. Jika gejala memburuk, segera konsultasi ke dokter.",
        'alergi': "Hindari pemicu alergi, konsumsi antihistamin sesuai anjuran. Jika reaksi alergi parah, cari bantuan medis segera.",
        'depresi': "Segera konsultasi dengan profesional kesehatan mental (psikiater/psikolog) untuk diagnosis dan penanganan lebih lanjut. Penting untuk mendapatkan dukungan.",
        'migrain': "Istirahat di tempat gelap dan tenang, minum obat pereda nyeri. Hindari pemicu migrain. Konsultasi dokter untuk penanganan yang tepat.",
        'flu_biasa': "Istirahat cukup, minum air putih, dan minum vitamin C. Gejala akan mereda dalam beberapa hari.",
        'gangguan_kecemasan': "Latih teknik relaksasi (napas dalam), hindari kafein. Segera cari bantuan psikolog/psikiater untuk penanganan dan terapi.",
        'diare': "Konsumsi cairan oralit atau banyak minum air putih, hindari makanan pedas/berlemak. Jika diare berlanjut atau disertai demam, konsultasi dokter.",
        'bronkitis': "Istirahat, minum banyak cairan, hindari asap. Dokter mungkin meresepkan obat batuk atau antibiotik jika ada infeksi bakteri.",
        'demam_berdarah': "Segera ke fasilitas kesehatan terdekat untuk pemeriksaan dan penanganan medis. Penting untuk pemantauan ketat.",
        'diabetes': "Konsultasi dengan dokter untuk pengelolaan gula darah, diet sehat, dan olahraga teratur. Patuhi pengobatan yang diberikan.",
        'eksim': "Gunakan pelembap secara teratur, hindari sabun keras dan pemicu gatal. Konsultasi dokter untuk salep atau obat yang sesuai.",
        'maag': "Hindari makanan pedas, asam, berlemak. Makan dalam porsi kecil tapi sering. Konsultasi dokter untuk obat antasida atau penekan asam lambung.",
        'stres': "Kelola stres dengan relaksasi, olahraga, dan waktu istirahat. Jika berlarut, cari bantuan profesional.",
        'anemia': "Konsumsi makanan kaya zat besi, atau suplemen zat besi sesuai anjuran dokter. Lakukan pemeriksaan untuk mengetahui penyebab anemia.",
        'serangan_jantung': "INI ADALAH KONDISI DARURAT MEDIS! SEGERA HUBUNGI LAYANAN GAWAT DARURAT (112/911) atau bawa ke IGD terdekat!"
    }
    return therapy_suggestions.get(predicted_diagnosis, "Diagnosis tidak dikenal. Mohon konsultasi ke dokter untuk penanganan lebih lanjut.")

print("--- Klasifikasi Teks: Diagnosa Penyakit dengan Rekomendasi Terapi ---")

# --- 1. Persiapan Data Simulasi Diagnosa Penyakit ---
data_diagnosa_terapi = {
    'symptom_description': [
        "Pasien mengeluh demam tinggi, batuk kering, dan nyeri tenggorokan selama tiga hari.", # influenza
        "Mengalami hidung tersumbat, bersin-bersin, mata gatal setelah terpapar debu.", # alergi
        "Merasa lelah terus-menerus, kehilangan minat pada hobi, sulit tidur selama berminggu-minggu.", # depresi
        "Sakit kepala hebat, mual, muntah, dan sensitif terhadap cahaya.", # migrain
        "Kulit gatal kemerahan, bengkak di beberapa area setelah makan seafood.", # alergi
        "Perasaan sedih yang mendalam, putus asa, dan perubahan nafsu makan signifikan.", # depresi
        "Gejala mirip flu tapi lebih ringan, dengan pilek dan sedikit demam.", # flu_biasa
        "Sering cemas tanpa alasan jelas, detak jantung cepat, tangan berkeringat.", # gangguan_kecemasan
        "Sakit perut hebat, diare, dan muntah berkali-kali.", # diare
        "Batuk berdahak, sesak napas, dan demam rendah.", # bronkitis
        "Gatal-gatal di seluruh tubuh, ruam merah, dan bengkak bibir setelah makan udang.", # alergi
        "Perasaan panik mendadak, jantung berdebar, dan kesulitan bernapas tanpa sebab jelas.", # gangguan_kecemasan
        "Demam dan nyeri otot serta sendi, disertai ruam merah.", # demam_berdarah
        "Sering buang air kecil, mudah haus, dan cepat lapar, berat badan turun tanpa sebab.", # diabetes
        "Mata kabur, sakit kepala, leher kaku, dan sensitif terhadap cahaya.", # migrain
        "Tidak bisa tidur, nafsu makan hilang, merasa tidak berharga.", # depresi
        "Sakit tenggorokan, batuk kering, hidung tersumbat, dan nyeri kepala ringan.", # flu_biasa
        "Demam tinggi, menggigil, nyeri badan, dan batuk yang parah.", # influenza
        "Kulit kering bersisik, gatal parah, terutama di malam hari.", # eksim
        "Nyeri ulu hati, mual, perut kembung, dan sering bersendawa.", # maag
        "Kesulitan berkonsentrasi, sering lupa, mudah tersinggung.", # stres
        "Lemas, pucat, pusing, dan detak jantung tidak teratur.", # anemia
        "Nyeri dada, sesak napas, berkeringat dingin, menjalar ke lengan kiri." # serangan_jantung
    ],
    'diagnosis': [
        'influenza', 'alergi', 'depresi', 'migrain', 'alergi', 'depresi', 'flu_biasa', 'gangguan_kecemasan',
        'diare', 'bronkitis', 'alergi', 'gangguan_kecemasan', 'demam_berdarah', 'diabetes', 'migrain',
        'depresi', 'flu_biasa', 'influenza', 'eksim', 'maag', 'stres', 'anemia', 'serangan_jantung'
    ]
}

df_diagnosa_terapi = pd.DataFrame(data_diagnosa_terapi)
print("\nDataset untuk Diagnosa Penyakit:")
display(df_diagnosa_terapi.head())

# --- 2. Pra-pemrosesan Teks ---
df_diagnosa_terapi['cleaned_symptom'] = df_diagnosa_terapi['symptom_description'].apply(preprocess_text)
print("\nDataset setelah pra-pemrosesan:")
display(df_diagnosa_terapi.head())

# --- 3. Ekstraksi Fitur (TF-IDF) ---
X_diagnosa_terapi = df_diagnosa_terapi['cleaned_symptom']
y_diagnosa_terapi = df_diagnosa_terapi['diagnosis']

X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X_diagnosa_terapi, y_diagnosa_terapi, test_size=0.3, random_state=42 # Removed stratify=y_diagnosa_terapi
)

print(f"\nJumlah data training diagnosa: {len(X_train_dt)}")
print(f"Jumlah data testing diagnosa: {len(X_test_dt)}")

tfidf_vectorizer_dt = TfidfVectorizer()
X_train_tfidf_dt = tfidf_vectorizer_dt.fit_transform(X_train_dt)
X_test_tfidf_dt = tfidf_vectorizer_dt.transform(X_test_dt)

print("Bentuk X_train_tfidf_dt:", X_train_tfidf_dt.shape)
print("Bentuk X_test_tfidf_dt:", X_test_tfidf_dt.shape)

# --- 4. Pelatihan Model (Multinomial Naive Bayes) ---
mnb_model_dt = MultinomialNB()
mnb_model_dt.fit(X_train_tfidf_dt, y_train_dt)
print("\nModel Multinomial Naive Bayes untuk diagnosa telah berhasil dilatih.")

# --- 5. Evaluasi Model ---
y_pred_dt = mnb_model_dt.predict(X_test_tfidf_dt)
accuracy_dt = accuracy_score(y_test_dt, y_pred_dt)
print(f"\nAkurasi Model Diagnosa (Naive Bayes): {accuracy_dt:.2f}")
report_dt = classification_report(y_test_dt, y_pred_dt, zero_division=0)
print("\nClassification Report Diagnosa (Naive Bayes):\n", report_dt)

# --- Interactive Form for Prediction and Therapy Suggestion ---

symptom_input_dt = widgets.Textarea(
    value='',
    placeholder='Masukkan deskripsi gejala Anda di sini (misal: demam tinggi, batuk kering)',
    description='Gejala:',
    disabled=False,
    layout=widgets.Layout(width='80%', height='100px')
)

predict_button_dt = widgets.Button(
    description='Prediksi Diagnosis & Terapi',
    disabled=False,
    button_style='success',
    tooltip='Klik untuk memprediksi diagnosis dan mendapatkan saran terapi',
    icon='medical_information'
)

output_area_dt = widgets.Output()

def on_predict_button_dt_clicked(b):
    with output_area_dt:
        clear_output()
        user_symptom_desc = symptom_input_dt.value
        if not user_symptom_desc.strip():
            print("Mohon masukkan deskripsi gejala Anda.")
            return

        print(f"Gejala yang Anda masukkan: '{user_symptom_desc}'")

        # Pra-pemrosesan teks
        preprocessed_symptom_dt = preprocess_text(user_symptom_desc)
        print(f"Teks setelah pra-pemrosesan: '{preprocessed_symptom_dt}'")

        if not preprocessed_symptom_dt:
            print("Tidak ada kata kunci yang tersisa setelah pra-pemrosesan. Tidak dapat memprediksi.")
            return

        # Mengubah teks yang telah diproses menjadi fitur TF-IDF
        X_user_symptom_tfidf_dt = tfidf_vectorizer_dt.transform([preprocessed_symptom_dt])

        # Melakukan prediksi
        predicted_diagnosis_dt = mnb_model_dt.predict(X_user_symptom_tfidf_dt)

        # Menghasilkan saran terapi
        therapy_message = generate_therapy(predicted_diagnosis_dt[0], user_symptom_desc)

        print(f"\nDiagnosis yang diprediksi: '{predicted_diagnosis_dt[0].upper()}'")
        print(f"Saran Terapi: {therapy_message}")

# Menghubungkan event klik tombol ke fungsi
predict_button_dt.on_click(on_predict_button_dt_clicked)

# Menampilkan widget
display(symptom_input_dt, predict_button_dt, output_area_dt)